In [ ]:
import lmstudio as lms
from tqdm import tqdm
model = lms.llm('google/gemma-3-12b')
num_features=12288
#num_features=2
for i in tqdm(range(3017,num_features)):
    with open(f'transcoder_training/feature{i}_acts_and_tokens.txt','r') as f:
        features=f.read()
    with open(f'feature_{i}_analysis.txt', 'w') as analysis:
        chat = lms.Chat("Here is a list of tokens and their activations for a certain feature ordered from the largest activation to the smallest. Try to decide whether the feature has some semantic interpretation and provide it.  The tokens with higher activations are more significant for the analysis, however all have an impact. Find the most common tokens, write them, and a general theme or themes if there are any. Single letters or fragments that do not correspond to words are parts of less common words, since the model was trained on a small vocabulary. Finish the analysis with a list containing the most common tokens and another list containing the common themes (just the themes, no descriptions).")
        chat.add_user_message(features)
        prediction = model.respond(chat)
        #print(prediction)
        analysis.write(str(prediction))


 25%|██▍       | 2287/9271 [18:22:07<56:05:39, 28.91s/it] 
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': 'dark', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': 'er', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': '\"', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received

KeyboardInterrupt: 

{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': ' describing', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': ' actions', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 2289, 'message': {'type': 'fragment', 'fragment': {'content': ' happening', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 2289, "event": "Received unhandled message {'type': 'channelSe

In [ ]:
#TODO: process the analyses into a datafreame or dict (mostly for labeling and coloring of concepcts)
# 1. process again with gemma using larger batches
# 2. load into dataframe

In [6]:
prompt='''Here is an analysis of tokens activating a certain feature. Return just the common themes listed at the end. Okay, let's analyze this token activation data to determine the feature's semantic interpretation. Below is an example answer:
Analysis:

Overall Impression:

The dominant theme is clearly religious prayer and supplication, with a strong emphasis on concepts of faith, divine intervention, and moral behavior. The presence of "God," "pray," "prayer," and related terms (prayed, praying) consistently at the top indicates this core meaning.  Following that, there's a significant cluster around themes of marriage/relationships and then a broader set of words relating to moral conduct, forgiveness, and divine permission.

Most Common Tokens:

Here's a list of the most frequent tokens, ordered by their activation scores (highest to lowest):

pray
God
prayed
prayer
married
prayers
Him
permit
lent
behave
punish
forgive
borrow
regreat
cast
Common Themes:

Religious Devotion/Prayer: This is the most prominent theme, encompassing supplication, worship, and connection with a higher power.
Marriage & Relationships: The frequent appearance of "married" suggests a focus on marital status or relationships in general.
Moral Conduct & Redemption: Words like "punish," "forgive," "permit," "behave," and "regret" point to themes of morality, consequences, and seeking redemption.
Divine Permission/Intervention: The presence of words such as “permit”, “lent” and “God” suggest a theme of divine intervention or permission.
Explanation of Less Common Tokens & Fragments:

Single Letters/Fragments: These are likely remnants from the model's training data, where it encountered incomplete words or parts of words that were not fully recognized. They don’t contribute significantly to the overall semantic interpretation.
"Him": Refers to God in a personal way.
"Lent": Could refer to lending something (materially) but also has religious connotations related to Lent, a period of fasting and repentance.
"Punish/Punished": Indicates consequences for actions or transgressions.
"Forgive": Relates to seeking pardon or absolution.
"Behave/Behaving": Suggests moral conduct and adherence to rules.
"Cast": Could refer to casting out, as in expulsion from a group or place.
Let me know if you'd like a deeper dive into any specific aspect of this analysis!

Answer:
Religious Devotion/Prayer
Marriage & Relationships
Moral Conduct & Redemption
Divine Permission/Intervention

Now do the same for the following analysis:

'''
with open(f'fixed_themes.txt', 'w') as themes:
    for i in tqdm(range(5300)):
        with open(f'feature_{i}_analysis.txt','r') as f:
            analysis=f.read()
            chat = lms.Chat(prompt)
            chat.add_user_message(analysis)
            prediction = model.respond(chat)
            #print(prediction)
            themes.write(f'\n{i}\n')
            themes.write(str(prediction))

100%|██████████| 5300/5300 [1:38:02<00:00,  1.11s/it]  


In [ ]:
batch_size=200

with open(f'fixed_themes.txt', 'r') as themes:
    k=0
    for line in themes:
        if k%batch_size==0:
            k=0
            collected=''
            chat = lms.Chat('You will be given a list of different themes. Find common themes between them and write a mapping associating a common theme to those that have them')
        if line.isnumeric():
            num=int(line)
        else:
            collected+=line
            if k==batch_size-1:
                chat.add_user_message(analysis)
                prediction = model.respond(chat)
                print(prediction)
            k+=1
            
            

Okay, I understand. You want me to analyze the provided "Common Themes" list and create a mapping that groups them based on broader, overarching themes. Here's my attempt:

**Mapping of Common Themes:**

Here's how I've grouped the original themes into larger categories, along with explanations for each grouping:

*   **Narrative & Description (Primary Theme):**
    *   **Description/Explanation ("about"):** This is a core element.  Everything else seems to orbit around describing something.
    *   **Narrative/Storytelling ("who", "when", "that"):** Directly relates to the construction and presentation of stories or accounts.
    *   **Events/Actions (falling, fell):** Events are fundamental components of narratives; they provide the 'what' that needs to be described.

*   **Process & Progression (Secondary Theme):**
    *   **Process/Sequence (planning, start, arrive):**  This highlights a sense of order and development within the narrative or description. It suggests things happen i

KeyboardInterrupt: 

{"channel_id": 13135, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 13135, 'message': {'type': 'promptProcessingProgress', 'progress': 0}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}


{"channel_id": 13135, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 13135, 'message': {'type': 'promptProcessingProgress', 'progress': 1}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 13135, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 13135, 'message': {'type': 'fragment', 'fragment': {'content': 'Okay', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 13135, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 13135, 'message': {'type': 'fragment', 'fragment': {'content': ',', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:1234/llm"}
{"channel_id": 13135, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 13135, 'message': {'type': 'fragment', 'fragment': {'content': ' I

In [8]:
def all_numbers_present(s, n):
    for i in range(1, n + 1):
        if str(i) not in s:
            return False
    return True

# Example usage
input_string = ''' Family & Relationships (Core Theme): 0, 7, 11, 19, 43, 57, 62, 72
Environment/Location (Core Theme): 1, 5, 8, 15, 20, 34, 38, 44, 54, 56, 60, 62
Narrative & Description (Core Theme): 2, 9, 10, 14, 17, 23, 24, 30, 40, 43, 53, 55, 65, 68, 73, 76
Action & Change (Core Theme): 3, 4, 6, 12, 13, 16, 21, 25, 28, 31, 39, 41, 46, 48, 52, 58, 61, 64, 70, 75, 80
Observation & Perception (Core Theme): 14, 23, 33, 37, 41, 49, 50, 53, 73, 78, 81, 82
Values and Qualities (Core Theme): 43, 45, 67, 74, 77, 78
Emotional Expression (Core Theme): 43, 56, 58, 69, 74, 76, 78'''
n=82

if 1 <= n <= 99:
    if all_numbers_present(input_string, n):
        print(f"All numbers from 1 to {n} are present as substrings.")
    else:
        print(f"Not all numbers from 1 to {n} are present as substrings.")
else:
    print("Invalid input: n must be between 1 and 99.")

Not all numbers from 1 to 82 are present as substrings.


In [9]:
def missing_numbers(s, n):
    missing = []
    for i in range(1, n + 1):
        if str(i) not in s:
            missing.append(i)
    return missing

if 1 <= n <= 99:
    missing = missing_numbers(input_string, n)
    if not missing:
        print(f"All numbers from 1 to {n} are present as substrings.")
    else:
        print(f"The following numbers are missing from the string: {missing}")
else:
    print("Invalid input: n must be between 1 and 99.")

The following numbers are missing from the string: [18, 22, 26, 27, 29, 32, 35, 36, 42, 47, 51, 59, 63, 66, 71, 79]


In [10]:
input_string='''
Family & Relationships (Core Theme): 0, 7, 11, 19, 26, 32
Environment/Location (Core Theme): 1, 5, 8, 15, 20, 34, 38, 40
Narrative & Description (Core Theme): 2, 9, 10, 14, 17, 23, 24, 30, 41, 42
Action & Change (Core Theme): 3, 4, 6, 12, 13, 16, 21, 25, 28, 31, 39, 40
Observation & Perception (Core Theme): 14, 23, 33, 37, 41
'''
n=42
if 1 <= n <= 99:
    missing = missing_numbers(input_string, n)
    if not missing:
        print(f"All numbers from 1 to {n} are present as substrings.")
    else:
        print(f"The following numbers are missing from the string: {missing}")
else:
    print("Invalid input: n must be between 1 and 99.")

The following numbers are missing from the string: [18, 22, 27, 29, 35, 36]


In [11]:
input_string='''Family & Relationships (Core Theme): 0, 7, 11, 19, 26, 32, 43, 57, 62, 72
Environment/Location (Core Theme): 1, 5, 8, 15, 20, 34, 38, 44, 54, 56, 60, 62, 79
Narrative & Description (Core Theme): 2, 9, 10, 14, 17, 23, 24, 30, 40, 43, 53, 55, 65, 68, 73, 76, 42
Action & Change (Core Theme): 3, 4, 6, 12, 13, 16, 21, 25, 28, 31, 39, 41, 46, 48, 52, 58, 61, 64, 70, 71, 75, 80
Observation & Perception (Core Theme): 14, 23, 33, 37, 41, 49, 50, 53, 73, 81, 82
Values and Qualities (Core Theme): 43, 45, 67, 74, 77, 78
Emotional Expression (Core Theme): 43, 56, 58, 69, 74, 76, 78'''
n=82
if 1 <= n <= 99:
    missing = missing_numbers(input_string, n)
    if not missing:
        print(f"All numbers from 1 to {n} are present as substrings.")
    else:
        print(f"The following numbers are missing from the string: {missing}")
else:
    print("Invalid input: n must be between 1 and 99.")

The following numbers are missing from the string: [18, 22, 27, 29, 35, 36, 47, 51, 59, 63, 66]
